Getting the forms from propublica.org

In [5]:
import requests
import time
import os

# Your API Key (if required by ProPublica in the future; currently not mandatory)
# api_key = "your_api_key"

# List of EINs to fetch Form 990s for
eins = [
    "91-1767292", "33-0647946", "84-4780735", "45-4329874", "83-3305529",
    "33-0587567", "94-2745941", "33-0145660", "94-2665367", "20-1832617",
    "47-5161428", "94-2216915", "23-7213237", "85-3432174", "33-0974992",
    "94-1693226", "22-3902362", "95-6377791", "33-0537412", "45-3042628",
    "95-2496099", "45-2639830", "95-3273023", "68-0120240", "95-2566791",
    "77-0565183", "82-4594246", "83-1159078", "01-0777856", "38-3891081",
    "94-2951488", "94-2788588", "88-1650309", "43-2050242", "30-0358349",
    "46-5112972"
]

# Base URL for ProPublica Nonprofit Explorer API
base_url = "https://projects.propublica.org/nonprofits/api/v2/organizations/"

# Directory to save the Form 990 PDFs
output_directory = "./form_990_pdfs/"

# Create the directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Loop through each EIN and fetch Form 990s
for ein in eins:
    try:
        # Fetch organization data using ProPublica API
        response = requests.get(f"{base_url}{ein}.json")
        response.raise_for_status()
        data = response.json()

        # Extract Form 990 PDF URLs
        filings = data.get("filings_with_data", [])
        for filing in filings:
            tax_year = filing.get("tax_prd_yr")
            pdf_url = filing.get("pdf_url")

            # If a PDF URL is available, download the file
            if pdf_url:
                print(f"Downloading Form 990 for EIN {ein}, Year {tax_year}")
                pdf_response = requests.get(pdf_url)
                pdf_response.raise_for_status()

                # Save the PDF file
                file_name = f"{output_directory}{ein}_Form990_{tax_year}.pdf"
                with open(file_name, "wb") as pdf_file:
                    pdf_file.write(pdf_response.content)

                print(f"Saved: {file_name}")

        # Pause to avoid rate limiting
        time.sleep(5)

    except Exception as e:
        print(f"Error fetching data for EIN {ein}: {e}")


Error fetching data for EIN 91-1767292: 403 Client Error: Forbidden for url: https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2024_prefixes_88-91%2F911767292_202206_990_2024011622239623.pdf
Error fetching data for EIN 33-0647946: 403 Client Error: Forbidden for url: https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_05_2023_prefixes_31-34%2F330647946_202112_990_2023051221225435.pdf
Error fetching data for EIN 84-4780735: 403 Client Error: Forbidden for url: https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_12_2023_prefixes_84-85%2F844780735_202212_990_2023120522064265.pdf
Error fetching data for EIN 45-4329874: 403 Client Error: Forbidden for url: https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2024_prefixes_45-46%2F454329874_202212_990_2024011022213965.pdf
Error fetching data for EIN 83-3305529: 403 Client Error: Forbidden for url: https://projects.propublica.org

Downloading the forms

In [7]:
import requests
import time

# List of EINs and their respective Form 990 URLs
form_990_links = {
    "91-1767292": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2024_prefixes_88-91%2F911767292_202206_990_2024011622239623.pdf", "year": 2022},
    "33-0647946": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_05_2023_prefixes_31-34%2F330647946_202112_990_2023051221225435.pdf", "year": 2021},
    "84-4780735": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_12_2023_prefixes_84-85%2F844780735_202212_990_2023120522064265.pdf", "year": 2022},
    "45-4329874": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2024_prefixes_45-46%2F454329874_202212_990_2024011022213965.pdf", "year": 2022},
    "83-3305529": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_11_2023_prefixes_82-83%2F833305529_202206_990_2023112021976780.pdf", "year": 2022},
    "33-0587567": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=02_2021_prefixes_31-34%2F330587567_202006_990_2021022617760783.pdf", "year": 2020},
    "94-2745941": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_11_2023_prefixes_91-94%2F942745941_202212_990_2023111621944828.pdf", "year": 2022},
    "33-0145660": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2023_prefixes_26-46%2F330145660_202112_990_2023011020756444.pdf", "year": 2021},
    "94-2665367": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2024_prefixes_91-94%2F942665367_202212_990_2024011022213051.pdf", "year": 2022},
    "20-1832617": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2024_prefixes_20-20%2F201832617_202212_990_2024011022211513.pdf", "year": 2022},
    "47-5161428": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2024_prefixes_47-47%2F475161428_202212_990_2024010322160468.pdf", "year": 2022},
    "94-2216915": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_10_2022_prefixes_91-95%2F942216915_202012_990O_2022102720601472.pdf", "year": 2020},
    "23-7213237": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_05_2023_prefixes_23-26%2F237213237_202112_990_2023051021197690.pdf", "year": 2021},
    "85-3432174": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_11_2023_prefixes_85-87%2F853432174_202212_990_2023112922026808.pdf", "year": 2022},
    "33-0974992": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2024_prefixes_31-34%2F330974992_202212_990_2024011022212996.pdf", "year": 2022},
    "94-1693226": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_08_2023_prefixes_81-95%2F941693226_202212_990EO_2023082821554295.pdf", "year": 2022},
    "22-3902362": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_12_2023_prefixes_22-23%2F223902362_202212_990_2023120522067973.pdf", "year": 2022},
    "95-6377791": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_12_2023_prefixes_94-99%2F956377791_202212_990EZ_2023121422119778.pdf", "year": 2022},
    "33-0537412": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_06_2023_prefixes_31-36%2F330537412_202206_990_2023060921417185.pdf", "year": 2022},
    "45-3042628": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_10_2023_prefixes_45-46%2F453042628_202212_990EZ_2023101821728707.pdf", "year": 2022},
    "95-2496099": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_09_2023_prefixes_87-95%2F952496099_202212_990_2023091321676151.pdf", "year": 2022},
    "45-2639830": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_11_2023_prefixes_45-46%2F452639830_202212_990EZ_2023112021969526.pdf", "year": 2022},
    "95-3273023": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=12_2020_prefixes_85-99%2F953273023_201912_990_2020120417466499.pdf", "year": 2019},
    "68-0120240": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_06_2023_prefixes_66-80%2F680120240_202206_990_2023060921427354.pdf", "year": 2022},
    "95-2566791": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_10_2023_prefixes_94-99%2F952566791_202212_990_2023103121822159.pdf", "year": 2022},
    "77-0565183": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_09_2023_prefixes_75-82%2F770565183_202212_990_2023091121659961.pdf", "year": 2022},
    "82-4594246": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=02_2021_prefixes_81-83%2F824594246_202006_990_2021021717714604.pdf", "year": 2020},
    "83-1159078": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=06_2021_prefixes_82-83%2F831159078_201912_990PF_2021060918305469.pdf", "year": 2019},
    "01-0777856": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_06_2023_prefixes_01-06%2F010777856_202206_990_2023060521373642.pdf", "year": 2022},
    "38-3891081": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2024_prefixes_37-39%2F383891081_202212_990_2024010922196607.pdf", "year": 2022},
    "94-2951488": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_11_2023_prefixes_91-94%2F942951488_202212_990_2023110821886222.pdf", "year": 2022},
    "94-2788588": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=02_2021_prefixes_92-95%2F942788588_201912_990_2021021717710423.pdf", "year": 2019},
    "88-1650309": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_01_2024_prefixes_86-88%2F881650309_202212_990_2024010522190160.pdf", "year": 2022},
    "43-2050242": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_11_2023_prefixes_41-45%2F432050242_202112_990_2023110321861046.pdf", "year": 2021},
    "30-0358349": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_04_2022_prefixes_26-33%2F300358349_202012_990EZ_2022042119929265.pdf", "year": 2020},
    "46-5112972": {"url": "https://projects.propublica.org/nonprofits/download-filing?path=download990pdf_10_2023_prefixes_46-46%2F465112972_202212_990PF_2023102421765175.pdf", "year": 2022},
}

output_directory = "./form_990_pdfs/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36"
}

# Loop through each EIN and download the Form 990
for ein, data in form_990_links.items():
    try:
        url = data["url"]
        year = data["year"]
        print(f"Downloading Form 990 for EIN {ein}, Year {year}")

        # Request the file
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Check for HTTP errors

        # Save the file
        file_name = f"{output_directory}{ein}_Form990_{year}.pdf"
        with open(file_name, "wb") as pdf_file:
            pdf_file.write(response.content)

        print(f"Saved: {file_name}")
        time.sleep(1)  # Pause to avoid overloading the server

    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for EIN {ein}, Year {year}: {e}")


Saved: ./form_990_pdfs/91-1767292_Form990_2022.pdf
Saved: ./form_990_pdfs/33-0647946_Form990_2021.pdf
Saved: ./form_990_pdfs/84-4780735_Form990_2022.pdf
Saved: ./form_990_pdfs/45-4329874_Form990_2022.pdf
Saved: ./form_990_pdfs/83-3305529_Form990_2022.pdf
Saved: ./form_990_pdfs/33-0587567_Form990_2020.pdf
Saved: ./form_990_pdfs/94-2745941_Form990_2022.pdf
Saved: ./form_990_pdfs/33-0145660_Form990_2021.pdf
Saved: ./form_990_pdfs/94-2665367_Form990_2022.pdf
Saved: ./form_990_pdfs/20-1832617_Form990_2022.pdf
Saved: ./form_990_pdfs/47-5161428_Form990_2022.pdf
Saved: ./form_990_pdfs/94-2216915_Form990_2020.pdf
Saved: ./form_990_pdfs/23-7213237_Form990_2021.pdf
Saved: ./form_990_pdfs/85-3432174_Form990_2022.pdf
Saved: ./form_990_pdfs/33-0974992_Form990_2022.pdf
Saved: ./form_990_pdfs/94-1693226_Form990_2022.pdf
Saved: ./form_990_pdfs/22-3902362_Form990_2022.pdf
Saved: ./form_990_pdfs/95-6377791_Form990_2022.pdf
Saved: ./form_990_pdfs/33-0537412_Form990_2022.pdf
Saved: ./form_990_pdfs/45-30426